# NimbusPay ticket triage: LoRA fine-tune of Qwen2.5-3B-Instruct

Pipeline: setup, data audit and cleaning, sequence length analysis, baselines, training, controlled experiments, error analysis, final model and test predictions.

Runs on a free Colab T4. Every run saves its adapter, logs and predictions to Google Drive under `MyDrive/nimbus_triage/runs/`.

## 0. Setup

Pinned library versions, seeds, paths and shared helpers. On Colab the data is read from `MyDrive/nimbus_triage/candidate_pack/`; locally it is read from `./candidate_pack/`.

In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

IN_COLAB = "google.colab" in sys.modules

PINS = {
    "transformers": "5.22.1",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
}
EXPECTED_TORCH = "2.14.1"


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


if IN_COLAB:
    missing = [f"{p}=={v}" for p, v in PINS.items() if installed(p) != v]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

print("colab:", IN_COLAB)
print(f"torch=={installed('torch')}", "" if (installed("torch") or "").startswith(EXPECTED_TORCH) else f"(expected {EXPECTED_TORCH})")
for pkg, want in PINS.items():
    have = installed(pkg)
    print(f"{pkg}=={have}", "" if have == want else f"(expected {want})")

In [ ]:
import csv
import json
import random
import time
import importlib.util
from datetime import datetime
from pathlib import Path

import numpy as np

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/nimbus_triage")
else:
    ROOT = Path.cwd()

PACK = ROOT / "candidate_pack"
DATA = PACK / "data"
RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SEED = 42
MAX_NEW_TOKENS = 200

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
CATEGORIES = ["payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"]
PRIORITIES = ["P1", "P2", "P3"]
CHANNELS = ["upi", "card", "netbanking", "wallet"]
LANGUAGES = ["en", "hi-en"]
SOURCES = ["app-chat", "email", "twitter", "ivr-transcript"]
TXN_FIELDS = ["amount_paise", "txn_date", "txn_id", "channel"]
NON_TXN_CATEGORIES = ["account_access", "kyc", "other"]

for p in [DATA / "train.jsonl", DATA / "dev.jsonl", DATA / "test_inputs.jsonl", PACK / "score.py", PACK / "SCHEMA.md"]:
    assert p.exists(), f"missing {p}"

print("root:", ROOT)
print("runs:", RUNS)

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def system_text(row):
    return row["messages"][0]["content"]


def user_text(row):
    return row["messages"][1]["content"]


def label_text(row):
    return row["messages"][2]["content"]


def ticket_body(row):
    return user_text(row).split("\n\n", 1)[-1]


def received_date(row):
    return datetime.strptime(user_text(row).split("\n", 1)[0].split()[1], "%Y-%m-%d").date()


def ticket_source(row):
    return user_text(row).split("\n")[1].split(": ", 1)[1].strip()


def canonical(label):
    return json.dumps({k: label[k] for k in KEYS}, ensure_ascii=False)


_spec = importlib.util.spec_from_file_location("nimbus_score", PACK / "score.py")
scorer = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(scorer)


def evaluate(gold_rows, pred_rows):
    return scorer.score(gold_rows, pred_rows)


def show_scores(res, prefixes=None):
    print(f"{'group':28} {'n':>5} {'valid':>7} {'exact':>7} {'fields':>7}")
    for name in sorted(res, key=lambda x: (x != "all", x)):
        if prefixes is not None and name != "all" and not name.startswith(tuple(prefixes)):
            continue
        r = res[name]
        print(f"{name:28} {r['n']:5d} {r['json_valid']:7.1%} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
    print("per-field accuracy:", {k: round(v, 3) for k, v in res["all"]["fields"].items()})


EXPERIMENTS_CSV = RUNS / "experiments.csv"
EXPERIMENT_COLUMNS = ["run", "change", "train_rows", "dev_exact", "dev_field_acc", "dev_json_valid",
                      "peak_gpu_gb", "train_minutes", "trainable_params", "timestamp"]


def log_experiment(record):
    record = {**record, "timestamp": datetime.now().isoformat(timespec="seconds")}
    new = not EXPERIMENTS_CSV.exists()
    with open(EXPERIMENTS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=EXPERIMENT_COLUMNS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow(record)
    return record


train_raw = load_jsonl(DATA / "train.jsonl")
dev = load_jsonl(DATA / "dev.jsonl")
test = load_jsonl(DATA / "test_inputs.jsonl")
print(len(train_raw), len(dev), len(test))
print(user_text(dev[0]))
print(label_text(dev[0]))

In [ ]:
import torch
import transformers


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)


set_seed()

print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    NATIVE_BF16 = props.major >= 8
    print("gpu:", props.name, f"{props.total_memory / 1e9:.1f} GB", f"| compute capability {props.major}.{props.minor}",
          "| native bf16:", NATIVE_BF16, "| training dtype:", "bf16" if NATIVE_BF16 else "fp16")
else:
    print("gpu: none")